# calories_burned_ver9 — Degree 4 Ridge vs MLP

| 모델 | 핵심 아이디어 |
|------|-------------|
| **Ridge d=4** | 다항식 차수 올려 더 복잡한 곡선 학습 |
| **MLP** | 신경망으로 적응적 기저함수 학습 |
| **앙상블** | 세 모델 최적 가중치 결합 |

## 1. 라이브러리 · 시드 · 상수

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import warnings; warnings.filterwarnings('ignore')

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS     = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 2. 데이터 로드

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")
print("Train:", train.shape, "  Test:", test.shape)

Train: (7500, 11)   Test: (7500, 10)


## 3. Feature Engineering (ver8 동일 — 수정금지)

In [3]:
def create_features_extended(df):
    df = df.copy()
    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Temp_diff']           = df['Body_Temperature(F)'] - 98.6
    df['Duration_bin']        = pd.cut(df['Exercise_Duration'],
                                       bins=[-np.inf,10,20,30,np.inf],
                                       labels=[0,1,2,3]).astype(int)
    df['Duration_x_BPM']     = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff']= df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff']     = df['BPM'] * df['Temp_diff']
    df['Intensity']           = df['Duration_x_BPM']
    df['Effort']              = df['Weight(lb)'] * df['Intensity']
    df['Duration_sq']         = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq']        = df['Temp_diff'] ** 2
    df['Dur_BPM_TempDiff']    = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']
    df['BPM_per_Duration']    = df['BPM'] / (df['Exercise_Duration'] + 1)
    df['TempDiff_per_Duration']= df['Temp_diff'] / (df['Exercise_Duration'] + 1)
    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI']                 = (703 * df['Weight(lb)'] / h2).fillna(0)
    df['Weight_x_Duration']   = df['Weight(lb)'] * df['Exercise_Duration']
    df['Log_BPM']             = np.log1p(df['BPM'])
    df['Log_Duration']        = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur']  = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])
    # ── 물리 공식 피처 ──────────────────────────────────────────────
    df['Weight_kg']           = df['Weight(lb)'] * 0.453592
    df['Height_cm']           = df['Height_Total_Inches'] * 2.54
    is_male                   = (df['Gender'] == 'M').astype(float)
    df['MaxHR']               = 208 - 0.7 * df['Age']
    df['HRR_pct']             = ((df['BPM'] - 60) / (df['MaxHR'] - 60)).clip(0, 1)
    keyser_m = (-55.0969 + 0.6309*df['BPM'] + 0.1988*df['Weight_kg'] + 0.2017*df['Age']) / 4.184
    keyser_f = (-20.4022 + 0.4472*df['BPM'] - 0.1263*df['Weight_kg'] + 0.0740*df['Age']) / 4.184
    df['Keyser_per_min']      = is_male * keyser_m + (1 - is_male) * keyser_f
    df['Keyser_total']        = df['Keyser_per_min'] * df['Exercise_Duration']
    df['Log_Keyser']          = np.log1p(np.clip(df['Keyser_total'], 0, None))
    bmr_m = 88.362  + 13.397*df['Weight_kg'] + 4.799*df['Height_cm'] - 5.677*df['Age']
    bmr_f = 447.593 +  9.247*df['Weight_kg'] + 3.098*df['Height_cm'] - 4.330*df['Age']
    df['BMR']                 = is_male * bmr_m + (1 - is_male) * bmr_f
    df['Age_x_BPM']           = df['Age'] * df['BPM']
    df['Age_x_Duration']      = df['Age'] * df['Exercise_Duration']
    df['Age_x_Effort']        = df['Age'] * df['Effort']
    return df

공식형 최소 피처 회귀 + 조합 스캔 + 제출 저장

In [11]:
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# =========================
# 0) 기본 설정
# =========================
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

y = train["Calories_Burned"].values.astype(float)

W = train["Weight(lb)"].values.astype(float)
D = train["Exercise_Duration"].values.astype(float)
B = train["BPM"].values.astype(float)
T = train["Body_Temperature(F)"].values.astype(float)

W_te = test["Weight(lb)"].values.astype(float)
D_te = test["Exercise_Duration"].values.astype(float)
B_te = test["BPM"].values.astype(float)
T_te = test["Body_Temperature(F)"].values.astype(float)

# 안정시 심박수(추정) - train에서만 구해 test에 그대로 사용 (누수 방지)
resting = np.quantile(B, 0.05)
B_adj = np.clip(B - resting, 0, None)
B_adj_te = np.clip(B_te - resting, 0, None)

# =========================
# 1) 공식형 후보 피처 만들기
# =========================
# 핵심: Weight × Duration × Intensity 계열 + 로그/비율/온도 보정
eps = 1e-6

cand = {
    # 기본 물리식
    "eff": W * D * B,                          # Effort (lb 기준)
    "eff_adj": W * D * B_adj,                  # resting 보정 Effort

    "intensity": D * B,
    "intensity_adj": D * B_adj,

    "work": W * D,
    "bpm_dur": B * D,

    # 로그/루트 변환 (선형화/스케일 안정)
    "log_eff": np.log1p(W * D * B),
    "log_eff_adj": np.log1p(W * D * B_adj),
    "log_intensity": np.log1p(D * B),
    "log_work": np.log1p(W * D),

    # 단변수도 포함(보정용)
    "w": W,
    "d": D,
    "b": B,
    "b_adj": B_adj,
    "temp": T,
    "temp_diff": T - 98.6,

    # 비율/정규화
    "b_per_d": B / (D + 1),
    "b_adj_per_d": B_adj / (D + 1),
    "eff_per_w": (W * D * B) / (W + 1),        # 대충 정규화
    "eff_per_d": (W * D * B) / (D + 1),
}

cand_te = {
    "eff": W_te * D_te * B_te,
    "eff_adj": W_te * D_te * B_adj_te,

    "intensity": D_te * B_te,
    "intensity_adj": D_te * B_adj_te,

    "work": W_te * D_te,
    "bpm_dur": B_te * D_te,

    "log_eff": np.log1p(W_te * D_te * B_te),
    "log_eff_adj": np.log1p(W_te * D_te * B_adj_te),
    "log_intensity": np.log1p(D_te * B_te),
    "log_work": np.log1p(W_te * D_te),

    "w": W_te,
    "d": D_te,
    "b": B_te,
    "b_adj": B_adj_te,
    "temp": T_te,
    "temp_diff": T_te - 98.6,

    "b_per_d": B_te / (D_te + 1),
    "b_adj_per_d": B_adj_te / (D_te + 1),
    "eff_per_w": (W_te * D_te * B_te) / (W_te + 1),
    "eff_per_d": (W_te * D_te * B_te) / (D_te + 1),
}

# DataFrame으로 변환
X_all = pd.DataFrame(cand)
X_all_te = pd.DataFrame(cand_te)

# 안전: inf/nan 처리
X_all = X_all.replace([np.inf, -np.inf], np.nan).fillna(0)
X_all_te = X_all_te.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Candidate features:", X_all.shape[1])

# =========================
# 2) 조합 스캔 (2~6개)
# =========================
alpha = 0.1  # 작은 feature set에선 적당한 규제가 보통 좋음 (나중에 최적화 가능)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=alpha, random_state=42))
])

feature_names = list(X_all.columns)

def cv_score(cols):
    oof = np.zeros(len(y))
    X = X_all[cols].values
    for tr_idx, va_idx in kf.split(X):
        pipe.fit(X[tr_idx], y[tr_idx])
        oof[va_idx] = pipe.predict(X[va_idx])
    oof = np.clip(oof, 0, None)
    return rmse(y, oof)

best = {"rmse": 1e18, "cols": None}

for k in range(2, 7):  # 2~6개
    for cols in tqdm(list(combinations(feature_names, k)), desc=f"scan k={k}", leave=False):
        score = cv_score(list(cols))
        if score < best["rmse"]:
            best = {"rmse": float(score), "cols": list(cols)}
            print("NEW BEST:", best)

print("\n✅ BEST combo:", best)

# =========================
# 3) 최적 조합으로 전체 학습 + test 예측 + 저장
# =========================
X_best = X_all[best["cols"]].values
X_best_te = X_all_te[best["cols"]].values

pipe.fit(X_best, y)
pred_te = pipe.predict(X_best_te)
pred_te = np.clip(pred_te, 0, None)

submission = pd.read_csv("sample_submission.csv")
submission["Calories_Burned"] = pred_te
out_name = "submit_formula_minfeat.csv"
submission.to_csv(out_name, index=False)

print("\n✅ saved:", out_name)
print("test stats mean/min/max:", float(pred_te.mean()), float(pred_te.min()), float(pred_te.max()))

Candidate features: 20


scan k=2:   7%|▋         | 13/190 [00:00<00:01, 129.55it/s]

NEW BEST: {'rmse': 20.21800603855433, 'cols': ['eff', 'eff_adj']}
NEW BEST: {'rmse': 13.565684428494258, 'cols': ['eff', 'intensity']}
NEW BEST: {'rmse': 13.379175709572424, 'cols': ['eff', 'intensity_adj']}
NEW BEST: {'rmse': 11.842655539359395, 'cols': ['eff_adj', 'intensity']}


scan k=3:   1%|▏         | 17/1140 [00:00<00:06, 164.58it/s]

NEW BEST: {'rmse': 11.525408697219662, 'cols': ['eff', 'eff_adj', 'intensity']}
NEW BEST: {'rmse': 11.52426013570681, 'cols': ['eff', 'intensity', 'work']}


...

scan k=6:  10%|▉         | 3756/38760 [00:26<03:44, 155.83it/s]

NEW BEST: {'rmse': 11.181781187571389, 'cols': ['eff', 'intensity', 'work', 'log_eff', 'log_intensity', 'w']}


scan k=6:  19%|█▉        | 7287/38760 [00:56<04:06, 127.56it/s]

NEW BEST: {'rmse': 11.181781187571378, 'cols': ['eff', 'work', 'bpm_dur', 'log_eff', 'log_intensity', 'w']}



✅ BEST combo: {'rmse': 11.181781187571378, 'cols': ['eff', 'work', 'bpm_dur', 'log_eff', 'log_intensity', 'w']}

✅ saved: submit_formula_minfeat.csv
test stats mean/min/max: 89.5277572822324 0.11013355223020938 276.8750124391195
